In [25]:
commented_datasets = [
    "stanford_cars",
    #"AID",
    #"ISIC-2019",
    #"Breast",
    #"Caltech-256",
    #"cifar100",
    #"mit_indoor",
    #"cub200",
    #"stanford-dogs",
    #"Aptos-2019",
    #"ISIC-2019",
    "oxford_iiit_pet",
    #"NABirds",
    #"dtd",
    #"fcvg_aircraft",
    #"sun397"
]


In [26]:
from torch.utils.data import DataLoader
import torch
from multi_dataset_adapter_dataset import AdapterArchitectureDataset, custom_collate

#dataset_names = ["Marathi-Sign-Language","lc25000","Brain-Tumour-MRI","brain-mri-dataset","BIOSCAN-30k","ArASL_Database_Grayscale","alzheimer_mri","deepweed","deepfashion","beans","Africa-Blood-Cell-Images-and-EHR-for-Cancer-Detection","Brain-Tumour-MRI","Blood","Aptos-2019"]
dataset_loadable=["idrid-disease-grading"]
dataset_notgood_performance = []
test_dataset_names = ["dtd","cifar100","Caltech-256","oxford_iiit_pet",  # general
                      "stanford_cars","stanford_dogs","cub200","nabirds", # fine graind
                        "","","","" # medical
                        "sun397","mit_indoor" # Scene
                        "AID" # Aerial
                        "uecfood" # food
                      ]
model_names = ["dinov2_ViT-B"]
#AID, Dementia_Dataset, BIOSCAN-30k: missing dataset encodings
#Ball_Screw_Drive_Classification: other error (check later)
#Breast : missing encodings for Dinov2-B
adapter_ds = AdapterArchitectureDataset(
    dataset_names=commented_datasets,
    model_names=model_names,
    include_null_architecture=False,
    work_dir="./work_dir",
)
len(adapter_ds)

Indexing dataset: stanford_cars
Indexing dataset: oxford_iiit_pet


2000

In [27]:
from meta_space_tools import ModelEncoder, DatasetEncoder, PerformancePredictor,  LossType
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Optional, List, Tuple


def _l2_normalize(x: torch.Tensor, dim: int = -1, eps: float = 1e-12) -> torch.Tensor:
    """L2 normalize a tensor along a dimension."""
    return x / (torch.norm(x, p=2, dim=dim, keepdim=True) + eps)


class DatasetEmbeddingBlender(nn.Module):
    """
    Small 2-layer network that learns to blend fixed PCA embeddings
    with learned adjustments.
    """
    
    def __init__(self, D: int, hidden_dim: int = 512, input_dim: Optional[int] = None, initial_blend_weight: float = 0.2):
        super().__init__()
        self.D = D
        
        # Small 2-layer network to learn adjustments
        if input_dim is None:
            input_dim = D

        self.adjustment_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            
            nn.Linear(hidden_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.1),

            nn.Linear(512, D)
        )
        
        # Learnable blending weight (initialized to favor fixed embeddings)
        # alpha controls: final = (1-alpha)*fixed + alpha*learned
        self.alpha = nn.Parameter(torch.tensor(initial_blend_weight), requires_grad=False)

    def forward(self, fixed_embedding: torch.Tensor, dataset_encodings: torch.Tensor) -> torch.Tensor:
        """
        Args:
            fixed_embedding: Fixed PCA embedding [batch_size, D]
            dataset_encodings: Dataset encodings [batch_size, D]

        Returns:
            Blended embedding [batch_size, D]
        """
        # Learn small adjustments
        learned_adjustment = self.adjustment_net(dataset_encodings)
        
        # Compute blending weight (constrain to [0, 1])
        alpha = torch.sigmoid(self.alpha)  # Maps to [0, 1]
        
        # Blend: weighted sum of fixed and adjusted
        # Scale learned adjustment to be relative to fixed embedding magnitude
        adjustment_scale = 0.1  # Keep adjustments small (10% of original magnitude)
        learned_embedding = _l2_normalize(learned_adjustment, dim=-1)
        
        
        # Weighted combination
        blended = (1 - alpha) * fixed_embedding + alpha * learned_embedding
        
        # L2 normalize to maintain unit sphere
        blended = _l2_normalize(blended, dim=-1)
        
        return blended, alpha


class MetaSpaceSystemHybridDatasets(nn.Module):
    """
    Meta-space system with HYBRID dataset embeddings:
    - Fixed PCA embeddings (non-learnable)
    - Small learned adjustments via 2-layer network
    - Weighted combination with more weight on fixed embeddings
    """
    
    def __init__(
        self,
        d_a: int,
        d_f: int,
        D: int = 256,
        fixed_dataset_embeddings: Optional[torch.Tensor] = None,  # [num_datasets, D]
        dataset_names: Optional[List[str]] = None,
        constrain_perf_output: bool = True,
        dataset_blend_hidden_dim: int = 512,
        initial_blend_weight: float = 0.2  # 0.2 = 80% fixed, 20% learned
    ):
        super().__init__()
        
        # Model encoder (trainable)
        self.model_encoder = ModelEncoder(d_a, d_f, D, hidden_dim=512)
        
        # Performance predictor (trainable)
        self.perf_predictor = PerformancePredictor(D, constrain_perf_output)
        
        # Dataset embedding blender (trainable) - one per dataset
        self.dataset_blender = DatasetEmbeddingBlender(D, dataset_blend_hidden_dim, input_dim=768,initial_blend_weight=initial_blend_weight)
        
        self.D = D
        
        # Register fixed dataset embeddings as buffer (NOT trainable)
        if fixed_dataset_embeddings is not None:
            self.register_buffer('fixed_dataset_embeddings', fixed_dataset_embeddings)
            self.dataset_names = dataset_names
            self.name_to_idx = {name: i for i, name in enumerate(dataset_names)} if dataset_names else {}
            print(f"Registered {len(fixed_dataset_embeddings)} fixed dataset embeddings")
            print(f"Dataset embedding shape: {fixed_dataset_embeddings.shape}")
            print(f"Initial blend weight: {initial_blend_weight:.2f} (learned) / {1-initial_blend_weight:.2f} (fixed)")
        else:
            self.fixed_dataset_embeddings = None
            self.dataset_names = None
            self.name_to_idx = {}
        
        # Set initial blend weight
        if hasattr(self, 'dataset_blender'):
            # Initialize to favor fixed embeddings
            # sigmoid^(-1)(0.2) ≈ -1.386
            import math
            init_logit = math.log(initial_blend_weight / (1 - initial_blend_weight))
            self.dataset_blender.alpha.data.fill_(init_logit)
        
        # Initialize trainable parameters
        self._init_weights()
    
    def _init_weights(self):
        """Initialize model encoder and performance predictor."""
        def _init_fn(m):
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=1.0)  # Use gain=1.0 for blender
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        
        self.model_encoder.apply(_init_fn)
        self.perf_predictor.apply(_init_fn)
        self.dataset_blender.apply(_init_fn)
        
        print("Initialized model encoder and performance predictor with Xavier normal (gain=2.0)")
        print("Initialized dataset blender with Xavier normal (gain=1.0)")
    
    def get_dataset_embedding(self, dataset_name: str) -> torch.Tensor:
        """Get fixed embedding for a dataset by name."""
        if dataset_name not in self.name_to_idx:
            raise ValueError(f"Unknown dataset: {dataset_name}. Available: {self.dataset_names}")
        idx = self.name_to_idx[dataset_name]
        return self.fixed_dataset_embeddings[idx]
    
    def get_fixed_dataset_embeddings_batch(self, dataset_names: List[str]) -> torch.Tensor:
        """Get fixed embeddings for a batch of dataset names."""
        indices = [self.name_to_idx[name] for name in dataset_names]
        return self.fixed_dataset_embeddings[indices]
    
    def get_blended_dataset_embeddings_batch(self, dataset_names: List[str], dataset_encodings: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Get blended dataset embeddings (fixed + learned) for a batch.
        
        Returns:
            blended_embeddings: [batch_size, D]
            alpha: Current blending weight (scalar)
        """
        # Get fixed embeddings
        fixed_embeddings = self.get_fixed_dataset_embeddings_batch(dataset_names)
        
        # Blend with learned adjustments
        blended_embeddings, alpha = self.dataset_blender(fixed_embeddings, dataset_encodings)

        return blended_embeddings, alpha
    
    def forward(
        self,
        arch_features: torch.Tensor,
        func_features: torch.Tensor,
        null_arch_features: torch.Tensor,
        dataset_encodings: torch.Tensor,
        dataset_names: List[str]
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass with hybrid dataset embeddings.
        
        Args:
            arch_features: Architecture features [batch_size, d_a]
            func_features: Functional features [batch_size, d_f]
            null_arch_features: Null architecture features [batch_size, d_f]
            dataset_names: List of dataset names for this batch
        
        Returns:
            Dictionary with embeddings and predictions
        """
        # Encode model (trainable)
        e_m = self.model_encoder(arch_features, func_features, null_arch_features)
        
        # Get hybrid dataset embeddings (fixed + learned)
        e_d, alpha = self.get_blended_dataset_embeddings_batch(dataset_names, dataset_encodings)
        
        # Predict performance
        y_hat = self.perf_predictor(e_m, e_d)
        
        return {
            'model_embedding': e_m,
            'dataset_embedding': e_d,  # Blended embeddings
            'performance_prediction': y_hat,
            'blend_alpha': alpha  # Track the blending weight
        }
    
    def compute_mse_loss(
        self,
        y_hat: torch.Tensor,
        y_true: torch.Tensor,
    ) -> torch.Tensor:
        """Simple MSE loss on predictions."""
        return F.mse_loss(y_hat.squeeze(), y_true.squeeze())
    
    def compute_rank_loss_pairwise_margin(
        self,
        e_m: torch.Tensor,
        e_d: torch.Tensor,
        perf_matrix: torch.Tensor,
        beta: float = 8.0,
        base_margin: float = 0.01,
        gap_power: float = 1.0,
        hard_k: int = 256
    ) -> torch.Tensor:
        """
        Pairwise margin ranking loss.
        Encourages model embeddings to have correct similarity ordering with dataset embeddings.
        """
        em = e_m
        ed = e_d
        
        sims = em @ ed.t()  # [M, N]
        perf = perf_matrix.clone()
        perf = torch.where(torch.isnan(perf), torch.full_like(perf, -1.0), perf)
        if perf.max() > 1.0 + 1e-6:
            perf = perf / 100.0
        
        total = e_m.new_zeros(())
        used = 0
        M, N = perf.shape
        
        for i in range(N):
            y = perf[:, i]
            mask = y >= 0
            if mask.sum() <= 1:
                continue
            
            s = sims[mask, i]
            p = y[mask]
            
            # Sort by perf desc
            vals, idx = torch.sort(p, descending=True)
            s = s[idx]
            
            # Top-1 as anchor positive
            t = min(1, vals.numel() - 1)
            pos = s[:t]
            neg = s[t:]
            
            if neg.numel() == 0:
                continue
            
            # Hard negative mining
            if hard_k and neg.numel() > hard_k:
                neg, _ = torch.topk(neg, k=hard_k, largest=True)
            
            # Adaptive margin
            gap = (vals[:t].mean() - vals[t:].mean()).clamp_min(0.0) ** gap_power
            margin = base_margin * (1.0 + gap)
            
            # Pairwise loss
            diff = pos.unsqueeze(1) - neg.unsqueeze(0)
            loss_i = F.softplus(beta * (margin - diff)).mean()
            
            total = total + loss_i
            used += 1
        
        return total / max(used, 1)
    
    def compute_model_embedding_regularization(
        self,
        e_m: torch.Tensor
    ) -> torch.Tensor:
        """
        Regularize only model embeddings to prevent collapse.
        """
        # Variance regularization
        var_m = torch.var(e_m, dim=0).mean()
        var_loss = torch.exp(-var_m)
        
        # Decorrelation regularization
        e_m_centered = e_m - e_m.mean(dim=0, keepdim=True)
        cov_m = (e_m_centered.T @ e_m_centered) / (e_m.shape[0] - 1)
        off_diag_m = cov_m - torch.diag(torch.diag(cov_m))
        corr_loss = off_diag_m.pow(2).mean()
        
        return var_loss + 0.5 * corr_loss
    
    def compute_blend_regularization(
        self,
        alpha: torch.Tensor,
        target_alpha: float = 0.2
    ) -> torch.Tensor:
        """
        Regularize blending weight to stay close to target (favoring fixed embeddings).
        
        Args:
            alpha: Current blending weight
            target_alpha: Target weight (0.2 = 20% learned, 80% fixed)
        """
        # Penalize deviation from target
        return (alpha - target_alpha) ** 2
    
    def compute_dataset_adjustment_regularization(
        self,
        dataset_names: List[str],
        dataset_encodings: torch.Tensor
    ) -> torch.Tensor:
        """
        Regularize dataset adjustments to be small.
        Encourages the network to make minimal changes to fixed embeddings.
        """
        # Get fixed embeddings
        fixed_embeddings = self.get_fixed_dataset_embeddings_batch(dataset_names)
        
        # Get adjustments
        adjustments = self.dataset_blender.adjustment_net(dataset_encodings)
        
        # L2 norm of adjustments (encourage small adjustments)
        adjustment_magnitude = torch.norm(adjustments, p=2, dim=1).mean()
        
        return adjustment_magnitude
    
    def compute_dataset_separation_loss(
        self,
        e_d: torch.Tensor,           # [N, D] dataset embeddings
        dataset_names: List,         # [N] list of dataset names
        min_distance: float = 0.5,   # minimum desired distance between unique datasets
        temperature: float = 0.1,    # controls sharpness of the penalty
        use_cosine: bool = True,     # use cosine distance vs euclidean
    ) -> torch.Tensor:
        """
        Regularizer to prevent unique datasets from collapsing too close together.
        Penalizes dataset pairs that are closer than min_distance.
        
        Args:
            e_d: Dataset embeddings [N, D] 
            dataset_names: List of dataset names for each embedding
            min_distance: Minimum desired distance between unique datasets
            temperature: Controls sharpness of penalty (lower = sharper)
            use_cosine: If True, use cosine distance; if False, use L2
        
        Returns:
            Scalar loss value
        """
        device, dtype = e_d.device, e_d.dtype
        
        # Step 1: Average embeddings for same datasets (deduplicate)
        unique_names = list(dict.fromkeys(dataset_names))  # Preserve order
        unique_embeddings = []
        
        for name in unique_names:
            # Find all indices for this dataset
            indices = [i for i, n in enumerate(dataset_names) if n == name]
            # Average the embeddings for this dataset
            avg_embedding = e_d[indices].mean(dim=0, keepdim=True)
            unique_embeddings.append(avg_embedding)
        
        if len(unique_embeddings) <= 1:
            return torch.zeros((), device=device, dtype=dtype)
        
        # Stack unique embeddings
        e_d_unique = torch.cat(unique_embeddings, dim=0)  # [K, D]
        K = e_d_unique.size(0)
        
        if use_cosine:
            # L2 normalize for cosine similarity
            e_d_norm = _l2_normalize(e_d_unique, dim=1)
            # Cosine similarity matrix
            sim_matrix = e_d_norm @ e_d_norm.t()  # [K, K], range [-1, 1]
            # Convert to distance (0 = identical, 2 = opposite)
            dist_matrix = 1.0 - sim_matrix  # [K, K], range [0, 2]
        else:
            # Euclidean distance
            dist_matrix = torch.cdist(e_d_unique, e_d_unique, p=2)
        
        # Mask diagonal (self-distances)
        mask = 1.0 - torch.eye(K, device=device, dtype=dtype)
        
        # Compute penalties for pairs that are too close
        # Using a soft margin: exp(-(dist - min_distance)/temperature)
        # This creates a smooth penalty that increases as distance decreases below min_distance
        violations = torch.exp(-(dist_matrix - min_distance) / temperature)
        
        # Only penalize distances below the threshold
        penalties = violations * (dist_matrix < min_distance).float()
        
        # Apply mask and compute mean penalty
        loss = (penalties * mask).sum() / (mask.sum() + 1e-8)
        
        return loss
    
    def compute_loss(
        self,
        outputs: Dict[str, torch.Tensor],
        batch_items: Dict,
        perf_matrix: torch.Tensor,
        weights: Optional[Dict[str, float]] = None,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Compute total loss with hybrid dataset embeddings.
        """
        default_weights = {
            'mse': 1.0,
            'rank': 0.5,
            'model_reg': 0.1,
            'blend_reg': 0.05,  # Regularize blend weight
            'adjustment_reg': 0.02,  # Regularize adjustment magnitude
            'dataset_separation': 0.1,  # Regularize dataset separation
        }
        if weights:
            default_weights.update(weights)
        
        e_m = outputs['model_embedding']
        e_d = outputs['dataset_embedding']  # Blended embeddings
        y_hat = outputs['performance_prediction']
        alpha = outputs['blend_alpha']
        
        loss_dict = {}
        perf_matrix = perf_matrix.to(e_m.device, dtype=e_m.dtype) / 100.0
        
        # MSE Loss
        y_true = batch_items['performance'].to(e_m.device) / 100.0
        if y_true.dim() == 1:
            y_true = y_true.unsqueeze(1)
        
        mse_loss = self.compute_mse_loss(y_hat, y_true)
        loss_dict['mse'] = float(mse_loss.detach())
        
        # Ranking Loss
        rank_loss = self.compute_rank_loss_pairwise_margin(e_m, e_d, perf_matrix)
        loss_dict['rank'] = float(rank_loss.detach())
        
        # Model embedding regularization
        model_reg = self.compute_model_embedding_regularization(e_m)
        loss_dict['model_reg'] = float(model_reg.detach())
        
        # Blend weight regularization (keep close to 0.2)
        blend_reg = self.compute_blend_regularization(alpha, target_alpha=0.2)
        loss_dict['blend_reg'] = float(blend_reg.detach())
        loss_dict['blend_alpha'] = float(alpha.detach())  # Track current alpha
        
        # Dataset adjustment regularization (keep adjustments small)
        #adjustment_reg = self.compute_dataset_adjustment_regularization(batch_items['dataset'])
        #loss_dict['adjustment_reg'] = float(adjustment_reg.detach())

        # compute dataset separation loss
        dataset_separation_loss = self.compute_dataset_separation_loss(
            e_d,
            batch_items['dataset'],
            min_distance=0.5,
            temperature=0.1,
            use_cosine=True
        )
        loss_dict['dataset_separation'] = float(dataset_separation_loss.detach())
        
        # Total
        total_loss = (default_weights['mse'] * mse_loss + 
                     default_weights['rank'] * rank_loss + 
                     default_weights['model_reg'] * model_reg +
                     default_weights['blend_reg'] * blend_reg +
                     default_weights['dataset_separation'] * dataset_separation_loss
)
        loss_dict['total'] = float(total_loss.detach())
        
        return total_loss, loss_dict

In [28]:
import torch
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import json

def load_embedding_pipeline(
    load_dir: str,
    name: str = "dataset_embeddings",
    device: str = 'cpu'
) -> Tuple[torch.Tensor, List[str], Dict]:
    """
    Load all components of the embedding pipeline for inference.
    
    Args:
        load_dir: Directory containing saved files
        name: Base name of saved files
        device: Device to load tensors to ('cpu' or 'cuda')
    
    Returns:
        fixed_embeddings: The loaded embeddings tensor [num_datasets, target_dim]
        dataset_names: List of dataset names
        loaded_info: Dictionary containing all loaded components and metadata
    """
    load_path = Path(load_dir)
    
    if not load_path.exists():
        raise FileNotFoundError(f"Load directory does not exist: {load_path}")
    
    print(f"\nLoading embedding pipeline from: {load_path}")
    
    # 1. Load embeddings and dataset names
    embeddings_path = load_path / f"{name}_embeddings.pt"
    if not embeddings_path.exists():
        raise FileNotFoundError(f"Embeddings file not found: {embeddings_path}")
    
    embeddings_data = torch.load(embeddings_path, map_location=device)
    fixed_embeddings = embeddings_data['embeddings'].to(device)
    dataset_names = embeddings_data['dataset_names']
    print(f"✓ Loaded embeddings of shape: {fixed_embeddings.shape}")
    print(f"✓ Loaded {len(dataset_names)} dataset names")
    
    # 2. Load metadata
    metadata_path = load_path / f"{name}_metadata.json"
    metadata = {}
    if metadata_path.exists():
        with open(metadata_path, 'r') as f:
            metadata = json.load(f)
        print(f"✓ Loaded metadata")
    else:
        print("⚠ Warning: Metadata file not found")
    
    # 3. Load sklearn models (PCA, scaler, MDS)
    sklearn_path = load_path / f"{name}_sklearn_models.pkl"
    sklearn_models = {}
    if sklearn_path.exists():
        with open(sklearn_path, 'rb') as f:
            sklearn_models = pickle.load(f)
        print(f"✓ Loaded sklearn models:")
        if 'pca' in sklearn_models:
            print(f"  - PCA (n_components={sklearn_models['pca'].n_components_})")
        if 'scaler' in sklearn_models:
            print(f"  - StandardScaler")
        if 'mds' in sklearn_models:
            print(f"  - MDS")
    else:
        print("⚠ Warning: sklearn models file not found")
    
    # 4. Load FID matrix if it exists
    fid_path = load_path / f"{name}_fid_matrix.npy"
    fid_matrix = None
    if fid_path.exists():
        fid_matrix = np.load(fid_path)
        print(f"✓ Loaded FID matrix of shape: {fid_matrix.shape}")
    
    # 5. Combine everything into info dictionary
    loaded_info = {
        # Sklearn models
        'pca': sklearn_models.get('pca'),
        'scaler': sklearn_models.get('scaler'),
        'mds': sklearn_models.get('mds'),
        
        # FID matrix
        'fid_matrix': fid_matrix,
        
        # Metadata
        'embedding_dim': metadata.get('embedding_dim'),
        'num_datasets': metadata.get('num_datasets'),
        'fid_weight': metadata.get('fid_weight', 0.0),
        'use_mds': metadata.get('use_mds', False),
        'target_dim': metadata.get('target_dim'),
        'n_components': metadata.get('n_components'),
        'variance_explained': metadata.get('variance_explained'),
        'total_variance_explained': metadata.get('total_variance_explained'),
        'fid_alignment_loss': metadata.get('fid_alignment_loss'),
        'has_pca': metadata.get('has_pca', False),
        'has_scaler': metadata.get('has_scaler', False),
        'has_mds': metadata.get('has_mds', False),
        'has_fid_matrix': metadata.get('has_fid_matrix', False),
    }
    
    print(f"\n✓ Pipeline loaded successfully!")
    return fixed_embeddings, dataset_names, loaded_info
def load_and_prepare_inference(
    checkpoint_path: str,
    embeddings_path: str,
    device: str = 'mps'
) -> Tuple[torch.nn.Module, Dict, Dict]:
    """
    Load model checkpoint and embedding pipeline for inference.
    
    Args:
        checkpoint_path: Path to model checkpoint (.pth file)
        embeddings_path: Path to embeddings directory
        device: Device to load model on
    
    Returns:
        model: Loaded model
        embeddings_info: Dictionary with embeddings and dataset info
        pipeline_info: Dictionary with pipeline components
    """
    print(f"\nLoading model checkpoint from: {checkpoint_path}")
    print(f"Loading embeddings from: {embeddings_path}")
    
    # 1. Load model checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    # Create model instance
    model = MetaSpaceSystemHybridDatasets(
        d_a=72,
        d_f=768,
        D=256,
        fixed_dataset_embeddings=None,  # Will be loaded from embeddings
        dataset_names=None,
        constrain_perf_output=True,
        dataset_blend_hidden_dim=512,
        initial_blend_weight=checkpoint.get('initial_blend_weight', 0.2)
    ).to(device)

    checkpoint['model_state_dict'].pop('fixed_dataset_embeddings')  # Remove embeddings to save memory
    print(checkpoint['model_state_dict'].keys())

    #model.dataset_blender.load_state_dict(checkpoint['t'])

    # Load state dict
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    model.eval()
    
    # 2. Load embeddings pipeline
    fixed_embeddings, dataset_names, pipeline_info = load_embedding_pipeline(
        embeddings_path,
        name="dataset_embeddings",
        device=device
    )
    
    # 3. Set fixed embeddings in model
    model.fixed_dataset_embeddings = fixed_embeddings
    model.dataset_names = dataset_names
    model.name_to_idx = {name: i for i, name in enumerate(dataset_names)}
    
    # 4. Get pre-computed embeddings if available
    embeddings_info = checkpoint.get('embeddings', {})
    
    print(f"\nLoaded successfully:")
    print(f"  Model epoch: {checkpoint['epoch']}")
    print(f"  Fixed embeddings shape: {fixed_embeddings.shape}")
    print(f"  Number of datasets: {len(dataset_names)}")
    if embeddings_info:
        print(f"  Pre-computed embeddings available for {len(embeddings_info.get('datasets', []))} datasets")
    
    return model, embeddings_info, pipeline_info

def compute_similarity_to_models(
    query_dataset_embedding: torch.Tensor,
    embeddings_info: Dict,
    k: int = 5
) -> List[Tuple[str, float]]:
    """
    Find most similar known datasets to a query dataset embedding.
    
    Args:
        query_dataset_embedding: New dataset embedding [D]
        embeddings_info: Dictionary with known dataset embeddings
        k: Number of nearest neighbors to return
    
    Returns:
        List of (dataset_name, similarity) tuples
    """
    # Get all known dataset embeddings
    known_models = embeddings_info['models']
    
    
    similarities = []
    for i, model in enumerate(known_models):
        #print(f"Processing model {i+1}/{len(known_models)}: {model['dataset_name']}")
        dataset_name = model['dataset_name']
        model_embed = model['model_embedding'].to(device=query_dataset_embedding.device)
        architecture_vector = model['architecture_vector']
        performnace=model['performance']
        predicted_performance=model['predicted_performance']

        
        # Compute cosine similarity
        similarity = F.cosine_similarity(
            query_dataset_embedding.unsqueeze(0),
            model_embed.unsqueeze(0)
        ).item()

        similarities.append((architecture_vector, dataset_name, similarity, performnace, predicted_performance))

    # Sort by similarity descending
    similarities.sort(key=lambda x: x[2], reverse=True)

    # check if top model similarity is low (less than 0.5 ) than do similarity between datasets, select closest dataset and use it as reference by doing similarity again against models
    if similarities[0][2] < 0.5:
        print(f"Top model similarity is low ({similarities[0][2]:.4f}), using dataset similarity.")
        # Compute dataset-to-dataset similarity
        dataset_similarities = []
        for i, dataset in enumerate(embeddings_info['datasets']):
            dataset_embed = dataset['dataset_embedding'].to(device=query_dataset_embedding.device)
            similarity = F.cosine_similarity(
                query_dataset_embedding.unsqueeze(0),
                dataset_embed.unsqueeze(0)
            ).item()
            dataset_similarities.append((dataset['dataset_name'],dataset['dataset_embedding'], similarity))

        # Sort by similarity descending
        dataset_similarities.sort(key=lambda x: x[2], reverse=True)
        # Use closest dataset as reference
        reference_dataset = dataset_similarities[0][0]
        # Get models associated with reference dataset
        reference_models = embeddings_info['models']
        ref_dataset_embed=dataset_similarities[0][1]
        print(f"Closest dataset: {reference_dataset} with similarity {dataset_similarities[0][2]:.4f}")
        # Get similarities to reference dataset's models
        similarities = []
        for model in reference_models:
            model_embed = model['model_embedding'].to(device=query_dataset_embedding.device)
            architecture_vector = model['architecture_vector']
            performance = model['performance']
            predicted_performance = model['predicted_performance']
            
            # Compute cosine similarity
            similarity = F.cosine_similarity(
                ref_dataset_embed.unsqueeze(0),
                model_embed.unsqueeze(0)
            ).item()
            
            similarities.append((architecture_vector, model['dataset_name'], similarity, performance, predicted_performance))

        # Sort by similarity descending
        similarities.sort(key=lambda x: x[2], reverse=True)

        print(f"Using {reference_dataset} as reference dataset.")

    return similarities[:k]

def inference_fixed_dataset_embeddings(
    dataset_encodings: torch.Tensor,  # [num_new_datasets, 768]
    saved_info: Dict,  # The info dict returned from training
    target_dim: int = 256,
) -> torch.Tensor:
    """
    Apply the saved transformation pipeline to new dataset encodings.
    
    Args:
        dataset_encodings: Raw dataset encodings for new datasets [num_new_datasets, 768]
        saved_info: Dictionary containing saved PCA, scaler, and other info from training
        target_dim: Target embedding dimension (should match training)
    
    Returns:
        fixed_embeddings: [num_new_datasets, target_dim]
    """
    print(f"Running inference on {dataset_encodings.shape[0]} datasets...")
    
    # Convert to numpy
    encodings_np = dataset_encodings.cpu().numpy()
    
    # Step 1: Apply saved normalization (if it was used during training)
    if saved_info.get('scaler') is not None:
        encodings_normalized = saved_info['scaler'].transform(encodings_np)
        print(f"Applied saved StandardScaler normalization")
    else:
        encodings_normalized = encodings_np
        print(f"No normalization applied")
    
    # Step 2: Apply saved PCA transformation
    pca = saved_info['pca']
    embeddings_pca = pca.transform(encodings_normalized)
    print(f"Applied PCA with {pca.n_components_} components")
    
    # Step 3: Handle FID-aware embeddings if they were used
    fid_weight = saved_info.get('fid_weight', 0)
    use_mds = saved_info.get('use_mds', False)
    
    if fid_weight > 0:
        print(f"\nWarning: Original training used FID-aware embeddings (weight={fid_weight}).")
        print(f"For inference on new datasets, we can only use the content-based PCA projection.")
        print(f"FID-based adjustment requires computing FID between new datasets and training datasets.")
        
        # Option: If you want to incorporate FID for new datasets, you would need:
        # 1. Compute FID between each new dataset and all training datasets
        # 2. Use MDS or similar to project into the FID space
        # This is left as a TODO since it requires additional computation
    
    # Step 4: Pad if needed
    n_components = embeddings_pca.shape[1]
    if n_components < target_dim:
        padding = np.zeros((embeddings_pca.shape[0], target_dim - n_components))
        embeddings_pca = np.concatenate([embeddings_pca, padding], axis=1)
        print(f"Padded embeddings from {n_components} to {target_dim} dimensions")
    
    # Step 5: L2 normalize to unit sphere
    norms = np.linalg.norm(embeddings_pca, axis=1, keepdims=True)
    embeddings_normalized = embeddings_pca / (norms + 1e-8)
    
    print(f"\nFinal embedding shape: {embeddings_normalized.shape}")
    print(f"L2 norms: min={np.linalg.norm(embeddings_normalized, axis=1).min():.4f}, "
          f"max={np.linalg.norm(embeddings_normalized, axis=1).max():.4f}")
    
    # Convert back to torch
    fixed_embeddings = torch.from_numpy(embeddings_normalized).float()
    
    return fixed_embeddings

In [32]:
import pickle
# 1. Load model and embeddings
checkpoint_path = "checkpoints/meta-space_trial_7/meta-space_9_9.pth"
embeddings_path = "checkpoints/meta-space_trial_7"

model, embeddings_info, pipeline_info = load_and_prepare_inference(
        checkpoint_path,
        embeddings_path
)


Loading model checkpoint from: checkpoints/meta-space_trial_7/meta-space_9_9.pth
Loading embeddings from: checkpoints/meta-space_trial_7
Initialized model encoder and performance predictor with Xavier normal (gain=2.0)
Initialized dataset blender with Xavier normal (gain=1.0)
odict_keys(['model_encoder.arch_proj.weight', 'model_encoder.arch_proj.bias', 'model_encoder.func_proj.weight', 'model_encoder.func_proj.bias', 'model_encoder.func_up_proj.weight', 'model_encoder.func_up_proj.bias', 'model_encoder.null_proj.weight', 'model_encoder.null_proj.bias', 'model_encoder.shared_func_proj.weight', 'model_encoder.shared_func_proj.bias', 'model_encoder.mlp.0.weight', 'model_encoder.mlp.0.bias', 'model_encoder.mlp.1.weight', 'model_encoder.mlp.1.bias', 'model_encoder.mlp.4.weight', 'model_encoder.mlp.4.bias', 'model_encoder.mlp.5.weight', 'model_encoder.mlp.5.bias', 'model_encoder.mlp.8.weight', 'model_encoder.mlp.8.bias', 'perf_predictor.mlp.0.weight', 'perf_predictor.mlp.0.bias', 'perf_pred

In [ ]:
# cub200 4001

sample= adapter_ds[7001]
#dataset_encodings = torch.tensor(sample["dataset_encoding"]["encodings"]).to(device=model.fixed_dataset_embeddings.device).unsqueeze(0)
dataset_encodings = torch.tensor(sample["dataset_encoding"]["encodings"]).to(device=model.fixed_dataset_embeddings.device).unsqueeze(0)
print(f"New dataset encodings shape: {dataset_encodings.shape}")
print(f"New dataset names: {sample['dataset']}")
fixed_embeddings = inference_fixed_dataset_embeddings(
    dataset_encodings,
    pipeline_info,
    target_dim=256
)
fixed_embeddings = fixed_embeddings.to(device=model.fixed_dataset_embeddings.device)
# embed with model
model.dataset_blender.eval()
outputs = model.dataset_blender(fixed_embeddings, dataset_encodings)
blended_embeddings = outputs[0]
print(f"Blended dataset embeddings shape: {blended_embeddings.shape}")
# 3. Compute similarities
compute_similarity_to_models(
    blended_embeddings[0],
    embeddings_info,
    k=10 # Top 5 similar datasets
)

/var/folders/85/p_tp0xw910v6r25splyysl800000gp/T/ipykernel_4120/1014090901.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dataset_encodings = torch.tensor(sample["dataset_encoding"]["encodings"]).to(device=model.fixed_dataset_embeddings.device).unsqueeze(0)
/Users/lotfi.mecharbat/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/lotfi.mecharbat/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/lotfi.mecharbat/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/

New dataset encodings shape: torch.Size([1, 768])
New dataset names: NABirds
Running inference on 1 datasets...
Applied saved StandardScaler normalization
Applied PCA with 64 components
Padded embeddings from 64 to 256 dimensions

Final embedding shape: (1, 256)
L2 norms: min=1.0000, max=1.0000
Blended dataset embeddings shape: torch.Size([1, 256])


[(tensor([ 0.,  0.,  0.,  0., 32.,  0., 16.,  0., 16., 16.,  0.,  0.,  0.,  0.,
           0.,  0., 16., 16.,  0.,  0.,  0.,  0.,  2.,  0.,  2.,  2.,  2.,  2.,
           2.,  2.,  0.,  0.,  0.,  0.,  8.,  8.,  0.,  0.,  0.,  0.,  8.,  0.,
           0.,  0.,  0.,  4.,  0.,  0., 32., 32., 32., 32., 32., 32.,  2.,  2.,
           2.,  0.,  0.,  0.,  4.,  0.,  4.,  0.,  4.,  4., 16.,  0., 16., 16.,
           0.,  0.], device='mps:0'),
  'waterbirds',
  0.6125377416610718,
  tensor(93.6875, device='mps:0'),
  tensor([0.9370], device='mps:0')),
 (tensor([ 0.,  0.,  0.,  0.,  4.,  4.,  0.,  0.,  0.,  0., 32., 32.,  8.,  8.,
           8.,  8.,  8.,  8., 32., 32., 32., 32., 32., 32., 16.,  0., 16.,  0.,
           0.,  0.,  0.,  0.,  0.,  0., 32.,  0., 32.,  0., 32.,  0.,  0.,  0.,
           0.,  0.,  0.,  0.,  2.,  2.,  0.,  0.,  0.,  0.,  8.,  8., 16., 16.,
          16.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
           0.,  0.], device='mps:0'),
  'uecfood100'

In [33]:
import yaml
from pathlib import Path
# Create output directory if it doesn't exist
output_dir = Path("architecture_recommendations")
output_dir.mkdir(exist_ok=True)

# Process all datasets
for i in range(2):
        sample = adapter_ds[i*1000+1]
        dataset_name = sample['dataset']
        
        # Get recommendations
        dataset_encodings = torch.tensor(sample["dataset_encoding"]["encodings"]).to(device=model.fixed_dataset_embeddings.device).unsqueeze(0)
        fixed_embeddings = inference_fixed_dataset_embeddings(dataset_encodings, pipeline_info, target_dim=256)
        fixed_embeddings = fixed_embeddings.to(device=model.fixed_dataset_embeddings.device)
        
        outputs = model.dataset_blender(fixed_embeddings, dataset_encodings)
        blended_embeddings = outputs[0]
        
        # Get top 10 similar architectures
        recommendations = compute_similarity_to_models(blended_embeddings[0], embeddings_info, k=10)
        
        # For each recommended architecture, create a YAML file
        for j, (arch_vector, target_dataset, similarity, performance, predicted_performance) in enumerate(recommendations):
            output_data = {
                'dataset_a': dataset_name,
                'dataset_b': target_dataset,
                'arch': {
                    'vector': arch_vector.tolist(),
                    'similarity': float(similarity),
                    'performance': float(performance),
                    'predicted_performance': float(predicted_performance)
                }
            }
            
            # Save to YAML file
            filename = f"{dataset_name}_{target_dataset}_arch_{j}.yaml"
            with open(output_dir / filename, 'w') as f:
                yaml.dump(output_data, f, default_flow_style=None)



/var/folders/85/p_tp0xw910v6r25splyysl800000gp/T/ipykernel_4120/203124863.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dataset_encodings = torch.tensor(sample["dataset_encoding"]["encodings"]).to(device=model.fixed_dataset_embeddings.device).unsqueeze(0)
/Users/lotfi.mecharbat/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/lotfi.mecharbat/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/lotfi.mecharbat/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/

Running inference on 1 datasets...
Applied saved StandardScaler normalization
Applied PCA with 64 components
Padded embeddings from 64 to 256 dimensions

Final embedding shape: (1, 256)
L2 norms: min=1.0000, max=1.0000


/var/folders/85/p_tp0xw910v6r25splyysl800000gp/T/ipykernel_4120/203124863.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dataset_encodings = torch.tensor(sample["dataset_encoding"]["encodings"]).to(device=model.fixed_dataset_embeddings.device).unsqueeze(0)
/Users/lotfi.mecharbat/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/lotfi.mecharbat/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/lotfi.mecharbat/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/

Running inference on 1 datasets...
Applied saved StandardScaler normalization
Applied PCA with 64 components
Padded embeddings from 64 to 256 dimensions

Final embedding shape: (1, 256)
L2 norms: min=1.0000, max=1.0000


In [10]:

sample= adapter_ds[10001]
#dataset_encodings = torch.tensor(sample["dataset_encoding"]["encodings"]).to(device=model.fixed_dataset_embeddings.device).unsqueeze(0)
dataset_encodings = torch.tensor(sample["dataset_encoding"]["encodings"]).to(device=model.fixed_dataset_embeddings.device).unsqueeze(0)
print(f"New dataset encodings shape: {dataset_encodings.shape}")
print(f"New dataset names: {sample['dataset']}")
fixed_embeddings = inference_fixed_dataset_embeddings(
    dataset_encodings,
    pipeline_info,
    target_dim=256
)
fixed_embeddings = fixed_embeddings.to(device=model.fixed_dataset_embeddings.device)
# embed with model
model.dataset_blender.eval()
outputs = model.dataset_blender(fixed_embeddings, dataset_encodings)
blended_embeddings = outputs[0]
print(f"Blended dataset embeddings shape: {blended_embeddings.shape}")
# 3. Compute similarities
compute_similarity_to_models(
    blended_embeddings[0],
    embeddings_info,
    k=10 # Top 5 similar datasets
)

/var/folders/85/p_tp0xw910v6r25splyysl800000gp/T/ipykernel_4120/2959227094.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dataset_encodings = torch.tensor(sample["dataset_encoding"]["encodings"]).to(device=model.fixed_dataset_embeddings.device).unsqueeze(0)
/Users/lotfi.mecharbat/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/lotfi.mecharbat/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/lotfi.mecharbat/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/

New dataset encodings shape: torch.Size([1, 768])
New dataset names: sun397
Running inference on 1 datasets...
Applied saved StandardScaler normalization
Applied PCA with 64 components
Padded embeddings from 64 to 256 dimensions

Final embedding shape: (1, 256)
L2 norms: min=1.0000, max=1.0000
Blended dataset embeddings shape: torch.Size([1, 256])


[(tensor([ 8.,  0.,  0.,  8.,  0.,  0., 16.,  0., 16.,  0.,  0.,  0.,  2.,  0.,
           2.,  2.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
           0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
           0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 16.,
           0., 16., 16., 16., 16.,  0., 16., 16.,  0.,  0.,  0.,  0.,  0.,  0.,
          32., 32.], device='mps:0'),
  'mini-imagenet',
  0.6493749618530273,
  tensor(97.4375, device='mps:0'),
  tensor([0.9795], device='mps:0')),
 (tensor([ 0.,  0.,  0.,  0.,  0.,  0.,  8.,  8.,  8.,  0.,  0.,  0.,  8.,  8.,
           8.,  8.,  8.,  8., 16., 16.,  0.,  0.,  0.,  0.,  8.,  8.,  8.,  0.,
           0.,  0.,  4.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  8.,  8.,  8.,
           0.,  0.,  0.,  0., 16., 16.,  0.,  0.,  0.,  0.,  8.,  8.,  0.,  0.,
           0.,  0.,  0.,  8.,  0.,  0.,  0., 16., 16., 16.,  0.,  0.,  0.,  0.,
           0.,  0.], device='mps:0'),
  'WeatherN